# S4 · AndinaLog · Notebook 1 · Diagnóstico de Productos

Diagnostica `andinalog_productos.csv` sin modificar Bronze. Usa `quality_engine.py` y `catalogo_reglas_productos.json`, ubicados junto a este notebook. Las conversiones numéricas solo sirven para evaluar reglas y no se exportan como transformaciones. No hace EDA ni tratamiento Silver.

`producto_id` es la clave candidata, pendiente de confirmación de negocio. El diagnóstico distingue copias idénticas, conflictos de contenido para un mismo ID y colisiones detectables al quitar espacios y convertir a mayúsculas. No se definen reglas de consistencia categoría-temperatura sin un umbral de negocio aprobado; tampoco se inventan referencias externas.


## 1 · Configuración y lectura Bronze

En local, ejecuta desde una carpeta del proyecto. En Colab, ajusta la ruta de Drive.


In [7]:
from pathlib import Path
import hashlib
import os
import sys
import tempfile
import pandas as pd
sys.path.append(str(Path.cwd().parent))
from quality_engine import cargar_catalogo, diagnosticar, reportes

ENTORNO = "auto"  # auto, local, drive
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
CARPETA_DATASETS = "AndinaLog_03B_Bronce"
NOMBRE_CSV = "andinalog_productos.csv"
VERSION_DIAGNOSTICO = "GIAD-M3-S4-productos-diagnostico-v3"
RUTA_CATALOGO = Path("catalogo_reglas_productos.json")

def encontrar_raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets" / CARPETA_DATASETS / NOMBRE_CSV).is_file():
            return carpeta
    raise FileNotFoundError("No se encontró el CSV Bronze en la carpeta actual o sus padres")

def configurar_rutas(entorno):
    if entorno == "auto":
        entorno = "drive" if "google.colab" in sys.modules else "local"
    if entorno == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(RUTA_PROYECTO_DRIVE)
    elif entorno == "local":
        raiz = encontrar_raiz_local()
    else:
        raise ValueError("ENTORNO debe ser auto, local o drive")
    bronze = raiz / "datasets" / CARPETA_DATASETS / NOMBRE_CSV
    salidas = raiz / "proyecto-integrador" / "diagnostico" / "andinalog_productos" / "salidas"
    if not bronze.is_file():
        raise FileNotFoundError(bronze)
    return bronze, salidas

catalogo = cargar_catalogo(RUTA_CATALOGO)
RUTA_BRONZE, DIRECTORIO_SALIDAS = configurar_rutas(ENTORNO)
HASH_BRONZE = hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest()
df_bronze = pd.read_csv(RUTA_BRONZE, dtype="string", encoding="utf-8-sig", keep_default_na=False)
print("Bronze:", RUTA_BRONZE, "| filas:", len(df_bronze), "| SHA-256:", HASH_BRONZE)


Bronze: c:\Users\remrodri\Github\practicasNotebookColab\datasets\AndinaLog_03B_Bronce\andinalog_productos.csv | filas: 63 | SHA-256: 0ee6adc9905e0d41ec5a2646f64633cccbab53ac35ee66a95f9bd46687744eda


## 2 · Diagnóstico e incidencias

Cada incumplimiento tiene regla, dimensión, severidad, acción y valor original. `fila_bronze` empieza en 1 para las filas de datos del CSV.


In [8]:
df_diagnosticado, df_problemas, estado_reglas = diagnosticar(df_bronze, catalogo)
df_problemas["version_diagnostico"] = VERSION_DIAGNOSTICO
df_cuarentena = df_diagnosticado.loc[df_diagnosticado["en_cuarentena"]].copy()
display(df_problemas)
print("Filas en cuarentena:", len(df_cuarentena))


,fila_bronze,rule_id,dimension_calidad,severidad,accion,tipo_problema,columna_afectada,codigo_error,valor_original,detalle,version_catalogo,version_diagnostico
0,8,PROD-R008,CONFORMIDAD,CRITICAL,CUARENTENA,FORMATO,,FORMATO_INVALIDO,prod-008,Se espera PROD-### exactamente,1.0.0,GIAD-M3-S4-productos-diagnostico-v3
1,16,PROD-R008,CONFORMIDAD,CRITICAL,CUARENTENA,FORMATO,,FORMATO_INVALIDO,prod-016,Se espera PROD-### exactamente,1.0.0,GIAD-M3-S4-productos-diagnostico-v3
2,18,PROD-R004,COMPLETITUD,CRITICAL,CUARENTENA,FALTANTE,,FALTANTE,,Campo obligatorio vacío,1.0.0,GIAD-M3-S4-productos-diagnostico-v3
3,22,PROD-R008,CONFORMIDAD,CRITICAL,CUARENTENA,FORMATO,,FORMATO_INVALIDO,prod-022,Se espera PROD-### exactamente,1.0.0,GIAD-M3-S4-productos-diagnostico-v3
4,26,PROD-R009,VALIDEZ,CRITICAL,CUARENTENA,VALIDEZ,,VALOR_NO_RECONOCIDO,conjelado,"Categoría distinta de Fresco, Congelado o Seco",1.0.0,GIAD-M3-S4-productos-diagnostico-v3
5,28,PROD-R009,VALIDEZ,CRITICAL,CUARENTENA,VALIDEZ,,VALOR_NO_RECONOCIDO,conjelado,"Categoría distinta de Fresco, Congelado o Seco",1.0.0,GIAD-M3-S4-productos-diagnostico-v3
6,31,PROD-R008,CONFORMIDAD,CRITICAL,CUARENTENA,FORMATO,,FORMATO_INVALIDO,prod-031,Se espera PROD-### exactamente,1.0.0,GIAD-M3-S4-productos-diagnostico-v3
7,36,PROD-R009,VALIDEZ,CRITICAL,CUARENTENA,VALIDEZ,,VALOR_NO_RECONOCIDO,conjelado,"Categoría distinta de Fresco, Congelado o Seco",1.0.0,GIAD-M3-S4-productos-diagnostico-v3
8,46,PROD-R008,CONFORMIDAD,CRITICAL,CUARENTENA,FORMATO,,FORMATO_INVALIDO,prod-046,Se espera PROD-### exactamente,1.0.0,GIAD-M3-S4-productos-diagnostico-v3
9,49,PROD-R004,COMPLETITUD,CRITICAL,CUARENTENA,FALTANTE,,FALTANTE,,Campo obligatorio vacío,1.0.0,GIAD-M3-S4-productos-diagnostico-v3


Filas en cuarentena: 13


## 3 · Métricas y reportes

Los porcentajes usan el total Bronze como denominador. Una fila puede generar varias incidencias.


In [9]:
tablas_reporte = reportes(df_diagnosticado, df_problemas, estado_reglas, catalogo, RUTA_BRONZE.name, HASH_BRONZE)
tablas_reporte["metricas"].loc[len(tablas_reporte["metricas"])] = ["version_diagnostico", VERSION_DIAGNOSTICO]
for nombre in ["metricas", "por_regla", "por_columna", "por_dimension", "estado_reglas"]:
    print(nombre)
    display(tablas_reporte[nombre])


metricas


,metrica,valor
0,archivo_bronze,andinalog_productos.csv
1,sha256_bronze,0ee6adc9905e0d41ec5a2646f64633cccbab53ac35ee66...
2,version_catalogo,1.0.0
3,filas_bronze,63
4,filas_con_problemas,13
5,filas_en_cuarentena,13
6,filas_solo_warning,0
7,incidencias,14
8,incidencias_critical,14
9,incidencias_warning,0


por_regla


,rule_id,dimension_calidad,severidad,accion,tipo_problema,columna_afectada,codigo_error,incidencias,filas_afectadas,porcentaje_filas
0,PROD-R004,COMPLETITUD,CRITICAL,CUARENTENA,FALTANTE,,FALTANTE,2,2,3.17
1,PROD-R008,CONFORMIDAD,CRITICAL,CUARENTENA,FORMATO,,FORMATO_INVALIDO,6,6,9.52
2,PROD-R009,VALIDEZ,CRITICAL,CUARENTENA,VALIDEZ,,VALOR_NO_RECONOCIDO,3,3,4.76
3,PROD-R017,UNICIDAD,CRITICAL,CUARENTENA,UNICIDAD,,DUPLICADO,2,2,3.17
4,PROD-R019,UNICIDAD,CRITICAL,CUARENTENA,UNICIDAD,,COLISION_ID_NORMALIZADO,1,1,1.59


por_columna


,columna_afectada,incidencias,filas_afectadas,porcentaje_filas
0,,14,13,20.63


por_dimension


,dimension_calidad,incidencias,filas_afectadas,porcentaje_filas
0,COMPLETITUD,2,2,3.17
1,CONFORMIDAD,6,6,9.52
2,UNICIDAD,3,3,4.76
3,VALIDEZ,3,3,4.76


estado_reglas


,rule_id,estado,motivo,incidencias
0,PROD-R001,EJECUTADA,,0
1,PROD-R002,EJECUTADA,,0
2,PROD-R003,EJECUTADA,,0
3,PROD-R004,EJECUTADA,,2
4,PROD-R005,EJECUTADA,,0
5,PROD-R006,EJECUTADA,,0
6,PROD-R007,EJECUTADA,,0
7,PROD-R008,EJECUTADA,,6
8,PROD-R009,EJECUTADA,,3
9,PROD-R010,EJECUTADA,,0


## 4 · Comprobaciones y exportación

Los CSV de salida son productos del diagnóstico. Se reemplazan en cada ejecución; el Bronze y el notebook anterior no se sobrescriben.


In [10]:
pd.testing.assert_frame_equal(df_diagnosticado[catalogo["columnas_bronze"]], df_bronze)
assert len(df_diagnosticado) == len(df_bronze)
assert df_diagnosticado["fila_bronze"].is_unique
assert (df_diagnosticado["cantidad_problemas"] == df_diagnosticado["cantidad_critical"] + df_diagnosticado["cantidad_warning"]).all()
assert set(df_cuarentena["fila_bronze"]) == set(df_problemas.loc[df_problemas["accion"].eq("CUARENTENA"), "fila_bronze"])

def exportar_salidas(directorio, tablas, ruta_bronze, huella_inicial):
    if hashlib.sha256(ruta_bronze.read_bytes()).hexdigest() != huella_inicial:
        raise RuntimeError("El CSV Bronze cambió durante la ejecución")
    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}
    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix=".tmp_productos_", dir=directorio,
                                             encoding="utf-8-sig", newline="", delete=False) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)
        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)
    return list(temporales)

prefijo = "andinalog_productos_"
tablas_salida = {
    prefijo + "diagnosticado.csv": df_diagnosticado,
    prefijo + "problemas.csv": df_problemas,
    prefijo + "cuarentena.csv": df_cuarentena,
    prefijo + "reporte_calidad.csv": tablas_reporte["metricas"],
    prefijo + "reporte_por_regla.csv": tablas_reporte["por_regla"],
    prefijo + "reporte_por_columna.csv": tablas_reporte["por_columna"],
    prefijo + "reporte_por_dimension.csv": tablas_reporte["por_dimension"],
    prefijo + "estado_reglas.csv": tablas_reporte["estado_reglas"],
}
rutas_creadas = exportar_salidas(DIRECTORIO_SALIDAS, tablas_salida, RUTA_BRONZE, HASH_BRONZE)
for ruta in rutas_creadas:
    print(ruta)
print("Bronze intacto; diagnóstico exportado")


c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\diagnostico\andinalog_productos\salidas\andinalog_productos_diagnosticado.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\diagnostico\andinalog_productos\salidas\andinalog_productos_problemas.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\diagnostico\andinalog_productos\salidas\andinalog_productos_cuarentena.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\diagnostico\andinalog_productos\salidas\andinalog_productos_reporte_calidad.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\diagnostico\andinalog_productos\salidas\andinalog_productos_reporte_por_regla.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\diagnostico\andinalog_productos\salidas\andinalog_productos_reporte_por_columna.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\diagnostico\andinalog_productos\salidas\andinalog_p

## Alcance

El tratamiento de IDs mal formados, categorías no reconocidas y duplicados corresponde al Notebook 2. Antes de usar Productos Silver como referencia de IoT, debe aprobarse su resultado y registrar su versión. Una regla categoría-temperatura solo debe añadirse cuando exista una política de conservación confirmada.
